# 🧠 الخوارزمي — نموذجك الخاص على كولاب مجاني

هذا النوت بوك يشغّل **نموذجك الخاص باسم "الخوارزمي"** على GPU مجاني من Google Colab، ويطلّع لك **رابطاً عاماً** تربطه بموقعك — يصير عندك محرك AI ملكك 100%.

## كيف تستخدمه؟
1. اضغط `Shift+Enter` على كل خانة كود **بالترتيب** (من فوق لتحت)
2. الخانة الأخيرة تطلع لك:
   - `COLABC_URL` → الرابط العام (انسخه)
   - `COLABC_KEY` → المفتاح السري (انسخه)
3. حط القيمتين في `.env` عندك (أو لوحة Render) ثم افتح الموقع → بتلاقي النموذج باسم **Al-Khwarizmi Local**

> ⚠️ الرابط يتغير كل ما تفتح جلسة كولاب جديدة — تعيد النسخ واللصق. الجلسة المجانية تعيش عدة ساعات ثم تنقطع (طبيعي).

---

**اختيار النموذج:** الافتراضي `qwen3:8b` (ممتاز بالعربية + سريع على T4). بدائل:
- `gpt-oss:20b` → أقوى وأضخم لكن أبطأ تنزيلاً وتشغيلاً
- `qwen3:4b` → أخف وأسرع إن كان الجهاز ضعيف

In [ ]:
# ===== الخطوة 1: التثبيت والتشغيل (مرة واحدة لكل جلسة) =====
import os, subprocess, time

print("GPU المتاح:")
!nvidia-smi -L

# تثبيت ollama (يدير النموذج محلياً)
!curl -fsSL https://ollama.com/install.sh | sh

# تثبيت cloudflared (يعمل نفقاً عاماً آمناً)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

# تشغيل خادم ollama بالخلفية
if os.system("pgrep ollama > /dev/null") != 0:
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    time.sleep(4)
print("\nollama يعمل ✅")

In [ ]:
# ===== الخطوة 2: تنزيل النموذج (مرة واحدة لكل جلسة) =====
# غيّر الاسم إن أردت: gpt-oss:20b أو qwen3:4b
model_name = "qwen3:8b"
os.environ["OLLAMA_MODEL"] = model_name

print(f"تنزيل {model_name}... (ياخذ دقائق)")
!ollama pull {model_name}

# تسخين بسيط حتى أول رد على الموقع يكون سريعاً
print("تسخين النموذج...")
!ollama run {model_name} "مرحباً، أجب بكلمة واحدة"
print("النموذج جاهز ✅")

In [ ]:
# ===== الخطوة 3: بوابة "الخوارزمي" (وسيط OpenAI-مطابق + مفتاح سري) =====
import json, secrets, threading, time as _t, os
from flask import Flask, request, Response
import requests

# مفتاح سري عشوائي — لا تعطيه لأحد
TOKEN = secrets.token_hex(12)
print("🔑 مفتاحك السري (COLABC_KEY):", TOKEN)

REAL = os.environ.get("OLLAMA_MODEL", "qwen3:8b")
proxy = Flask("khwarizmi_proxy")

@proxy.route("/v1/models", methods=["GET"])
def list_models():
    if request.headers.get("Authorization") != f"Bearer {TOKEN}":
        return Response('{"error":"unauthorized"}', 401, content_type="application/json")
    return Response(json.dumps({
        "object": "list",
        "data": [{"id": "khwarizmi", "object": "model", "owned_by": "light-co", "permission": []}]
    }), 200, content_type="application/json")

@proxy.route("/v1/chat/completions", methods=["POST"])
def chat():
    if request.headers.get("Authorization") != f"Bearer {TOKEN}":
        return Response('{"error":"unauthorized"}', 401, content_type="application/json")
    body = request.get_json(force=True) or {}
    body.pop("stream", None)          # الموقع لا يستخدم التدفق
    body["model"] = REAL             # اسم النموذج الحقيقي داخل ollama
    try:
        r = requests.post("http://127.0.0.1:11434/v1/chat/completions", json=body, timeout=900)
        return Response(r.text, status=r.status_code, content_type="application/json")
    except Exception as e:
        return Response(json.dumps({"error": str(e)}), 502, content_type="application/json")

threading.Thread(target=lambda: proxy.run(host="127.0.0.1", port=11435, threaded=True), daemon=True).start()
_t.sleep(2)
print("البوابة شغالة على 127.0.0.1:11435 ✅")

In [ ]:
# ===== الخطوة 4: الرابط العام عبر Cloudflare =====
import subprocess, time, re, threading

if 'TOKEN' not in globals():
    import secrets
    TOKEN = secrets.token_hex(12)
    print("🔑 مفتاحك السري (COLABC_KEY):", TOKEN)

url_box = []
def _run():
    p = subprocess.Popen(
        ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:11435", "--no-autoupdate", "--loglevel", "info"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    pat = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
    for line in p.stdout:
        m = pat.search(line)
        if m and not url_box:
            url_box.append(m.group(0))
            break

threading.Thread(target=_run, daemon=True).start()
for _ in range(90):
    if url_box:
        break
    time.sleep(1)

print("=" * 62)
if url_box:
    URL = url_box[0]
    print(f"🌍 COLABC_URL = {URL}/v1")
    print(f"🔑 COLABC_KEY = {TOKEN}")
    print("=" * 62)
    print("انسخ القيمتين وحطهما في .env (أو لوحة Render) → افتح الموقع واختر النموذج:")
    print("   ▶ Al-Khwarizmi Local — نموذجك الخاص!")
    print("\n⚠️ الرابط جديد عند كل جلسة كولاب. والمفتاح يظل واحداً لكل جلسة.")
else:
    print("الرابط ما ظهر خلال 90 ثانية — شوف رسائل cloudflared فوق وأعد تشغيل هذه الخانة.")

## 📌 خلاصة الربط
```
COLABC_URL = https://xxxxxxxx.trycloudflare.com/v1
COLABC_KEY = مفتاحك السري
```

**في الموقع:**
- لو تشغّله محلياً: ضع القيمتين في `.env` وأعد تشغيل `python -u app.py`
- لو الموقع منشور على Render: لوحة Render → Environment → أضف المتغيرين → Save & Deploy

**متى يرجع الاحتياط تلقائياً؟** إذا أغلقت كولاب أو انقطعت الجلسة → موقعك ما يتأثر، يكبّر على المزوّدين الباقين (Gemini/OpenRouter/Groq/NVIDIA) ويعيد محاولة الخوارزمي المحلي تلقائياً.

> 🧠 ملاحظة صادقة: كولاب المجاني لأغراض تفاعلية — لا تستضف خدمة ضخمة للعموم عليه، استخدمه لنموذج موقعك الشخصي والتجريبي. وللاستقرار الدائم، الخيار الأقوى يبقى مزوّداً سحابياً مجانياً مع التسمية الحصرية (Al-Khwarizmi Flash).